# Sub-Question Query Engine & Multi-Document Agents

So far every index has covered the whole corpus at once. A different pattern: give each document its **own** index and `QueryEngineTool`, then let a `SubQuestionQueryEngine` break a cross-cutting question apart into one sub-question per relevant document, route each to the right tool, and synthesize the results — useful when documents are genuinely distinct topics rather than chunks of one bigger corpus.


**Step 1 — Setup.** Quiet the logs, load API keys, and set the global LLM/embedding models used throughout this notebook.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from httpx and llama_index.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Global defaults used by every index/query engine built below.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — One index and tool per anime.** Instead of one big index over all five documents, build a separate `VectorStoreIndex` for each anime and wrap each one in a `QueryEngineTool` with a name and description the sub-question engine can use to decide which tool answers which sub-question.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.tools import QueryEngineTool

# Maps each source file to a human-readable title used in the tool name/description.
anime_titles = {
    "naruto.txt": "Naruto",
    "dragon_ball.txt": "Dragon Ball",
    "solo_leveling.txt": "Solo Leveling",
    "death_note.txt": "Death Note",
    "demon_slayer.txt": "Demon Slayer",
}

query_engine_tools = []
for filename, title in anime_titles.items():
    # Load and index just this one file — a fully separate, single-document index.
    docs = SimpleDirectoryReader(input_files=[f"data/sample_docs/{filename}"]).load_data()
    doc_index = VectorStoreIndex.from_documents(docs)
    # Wrap the index's query engine as a Tool the sub-question engine can call.
    # The description is what the LLM reads to decide when to route to this tool.
    tool = QueryEngineTool.from_defaults(
        query_engine=doc_index.as_query_engine(),
        name=title.lower().replace(" ", "_"),
        description=f"Useful for answering questions specifically about {title}.",
    )
    query_engine_tools.append(tool)

print(f"Built {len(query_engine_tools)} per-anime query engine tools")

Built 5 per-anime query engine tools


**Step 3 — Ask one question that spans all five documents.** `SubQuestionQueryEngine` uses an LLM to break the broad question into one sub-question per tool, runs each sub-question against its own anime's tool, and synthesizes everything into one final answer. `verbose=True` prints each generated sub-question and its individual answer so you can see the decomposition happen live.


In [3]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.question_gen.llm_generators import LLMQuestionGenerator
from llama_index.llms.openai import OpenAI

# The question generator only makes one structured decision (how to split the
# query), but that decision needs to be reliable — gpt-4o-mini instead of the
# project-wide gpt-4.1-nano default, same reasoning as the router episode.
question_gen = LLMQuestionGenerator.from_defaults(llm=OpenAI(model="gpt-4o-mini"))

# Combines the question generator with all five per-anime tools — this object
# is what actually decomposes, routes, and re-synthesizes.
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=query_engine_tools,
    question_gen=question_gen,
    verbose=True,  # prints each sub-question and its individual answer as it runs
)

response = sub_question_engine.query("Compare how each protagonist grows stronger across all five series.")
print("\n--- FINAL SYNTHESIZED ANSWER ---")
print(response)

Generated 5 sub questions.
[naruto] Q: How does Naruto grow stronger throughout the series?
[dragon_ball] Q: How does Goku grow stronger throughout the Dragon Ball series?
[solo_leveling] Q: How does Sung Jin-Woo grow stronger in Solo Leveling?
[death_note] Q: How does Light Yagami grow in terms of intelligence and strategy in Death Note?
[demon_slayer] Q: How does Tanjiro Kamado grow stronger in Demon Slayer?
[solo_leveling] A: Sung Jin-Woo grows stronger by gaining access to a unique system that allows him to level up, complete quests, and increase his abilities. After a near-death experience in a hidden dungeon, he awakens as a "Player" with these powers. Throughout the series, he continuously enhances his strength by defeating enemies, summoning shadow soldiers, and progressing through various story arcs that showcase his increasing power and capabilities.
[naruto] A: Naruto grows stronger throughout the series by training under various mentors, learning new techniques such as the 

### Summary

- `SubQuestionQueryEngine` decomposes one broad question into one sub-question per relevant tool, runs them (in parallel by default), and synthesizes a single answer — the `verbose=True` trace above shows exactly which sub-question went to which anime's tool.
- This is a different pattern from `RouterQueryEngine` (Episode 5): the router picks **one** tool per question, while sub-question decomposition can fan a single question out to **several** tools at once when the question genuinely spans multiple documents.
